# Videos Pipeline

- Reference data
- One-time extraction
- Raw JSON → Volume
- Volume → Bronze Delta

In [0]:
# %pip install google-api-python-client python-dotenv
# %restart_python

## 1. Create Volume and incoming folder

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS youtube_content_intelligence.bronze.vol_videos;

In [0]:
dbutils.fs.mkdirs("/Volumes/youtube_content_intelligence/bronze/vol_videos/incoming/")

## 2. Extract video data

In [0]:
from src.extraction.videos import extract_videos

videos = extract_videos()

## 3. Write raw JSON to Volume

In [0]:
import json

volume_path = "/Volumes/youtube_content_intelligence/bronze/vol_videos/incoming/videos.json"

with open(volume_path, "w") as file:
    json.dump(videos, file, indent=2)

## 4. Verify raw JSON

In [0]:
display(dbutils.fs.ls("/Volumes/youtube_content_intelligence/bronze/vol_videos/incoming/"))

In [0]:
display(dbutils.fs.head("/Volumes/youtube_content_intelligence/bronze/vol_videos/incoming/videos.json", 1000))

## 5. Load raw JSON

In [0]:
df = (
    spark.read
    .option("multiLine", "true")
    .json("/Volumes/youtube_content_intelligence/bronze/vol_videos/incoming/videos.json")
)

In [0]:
df.printSchema()

## 6. Write to Bronze Delta

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("youtube_content_intelligence.bronze.brz_videos")

## 7. Validate Bronze table

In [0]:
%sql
SELECT *
FROM youtube_content_intelligence.bronze.brz_videos;

In [0]:
%sql
DESCRIBE TABLE youtube_content_intelligence.bronze.brz_videos;

In [0]:
%sql
SELECT COUNT(*) AS row_count
FROM youtube_content_intelligence.bronze.brz_videos;